In [ ]:
#Diable the warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf

wti = yf.download("CL=F", start="1983-01-01", end="2026-04-30")
brent = yf.download("BZ=F", start="1988-01-01", end="2026-05-02")

print(brent.tail())

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# =========================
# 1. دانلود داده
# =========================

wti = yf.download("CL=F", start="1983-01-01", end="2026-04-30")
brent = yf.download("BZ=F", start="1988-01-01", end="2026-05-02")

# =========================
# 2. فقط ستون Close + rename
# =========================

wti = wti[["Close"]].rename(columns={"Close": "WTI"})
brent = brent[["Close"]].rename(columns={"Close": "Brent"})

# =========================
# 3. merge بر اساس تاریخ
# =========================

data = pd.merge(wti, brent, left_index=True, right_index=True, how="inner")

# =========================
# 4. حذف NA
# =========================

data = data.dropna()

# =========================
# 5. ساخت بازدهی
# =========================

returns = np.log(data / data.shift(1)).dropna()
returns.columns = ["WTI_ret", "Brent_ret"]

print(returns.head())

# =========================
# 4. تابع آمار توصیفی
# =========================

def descriptive_stats(series):
    return pd.Series({
        "Mean": series.mean(),
        "Std": series.std(),
        "Min": series.min(),
        "Max": series.max(),
        "Skewness": skew(series),
        "Kurtosis": kurtosis(series),
        "JB_stat": jarque_bera(series)[0],
        "JB_pvalue": jarque_bera(series)[1]
    })

desc_table = returns.apply(descriptive_stats)

print("\n=== Descriptive Statistics ===")
print(desc_table.T)

# =========================
# 5. آزمون ADF (ریشه واحد)
# =========================

def adf_test(series):
    result = adfuller(series)
    return {
        "ADF_stat": result[0],
        "p-value": result[1]
    }

print("\n=== ADF Test ===")
for col in returns.columns:
    res = adf_test(returns[col])
    print(f"{col}: {res}")

# =========================
# 6. Ljung-Box (خودهمبستگی)
# =========================

print("\n=== Ljung-Box Test (lag=10) ===")
for col in returns.columns:
    lb = acorr_ljungbox(returns[col], lags=[10], return_df=True)
    print(f"\n{col}")
    print(lb)

# =========================
# 7. ARCH LM test
# =========================

print("\n=== ARCH LM Test ===")
for col in returns.columns:
    arch_test = het_arch(returns[col])
    print(f"{col}: LM Stat={arch_test[0]}, p-value={arch_test[1]}")

In [ ]:
import numpy as np
import pandas as pd

def parkinson_daily(df):
    log_hl = np.log(df['High'] / df['Low'])
    sigma_daily = np.sqrt((log_hl ** 2) / (4 * np.log(2)))
    return sigma_daily

In [ ]:
def yang_zhang_rolling(df, window):

    df = df.copy()

    log_ho = np.log(df['High'] / df['Open'])
    log_lo = np.log(df['Low'] / df['Open'])
    log_co = np.log(df['Close'] / df['Open'])

    log_oc = np.log(df['Open'] / df['Close'].shift(1))
    log_cc = np.log(df['Close'] / df['Close'].shift(1))

    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)

    sigma_o = log_oc.rolling(window).var()
    sigma_c = log_cc.rolling(window).var()
    sigma_rs = rs.rolling(window).mean()

    k = 0.34 / (1.34 + (window + 1)/(window - 1))

    yz_var = sigma_o + k * sigma_c + (1 - k) * sigma_rs

    return np.sqrt(yz_var)

In [ ]:
# فرض: df داری با OHLC
wti['YZ_vol'] =  yang_zhang_rolling(wti, window=30)



In [ ]:
wti['Parkinson_daily'] = parkinson_daily(wti)

In [ ]:
wti.columns = wti.columns.get_level_values(0)

wti=pd.DataFrame(wti[['YZ_vol','Parkinson_daily','Close']], columns=['YZ_vol','Parkinson_daily','Close'])
wti.head(2000)

In [ ]:
pip install xlrd==2.0.1

In [ ]:
gpr = pd.read_excel("data_gpr_daily_recent.xls")
gpr.to_csv("gpr.csv", index=False)
gpr = pd.read_csv("gpr.csv", parse_dates=['date'])
gpr=gpr[['date','GPRD','GPRD_ACT','GPRD_THREAT']]
gpr

In [ ]:
wti1=wti.reset_index()
wti2 = wti1.rename(columns={'Date': 'date'})
data = wti2.merge(gpr, on='date', how='inner')
data

In [ ]:
gpr1=gpr.set_index('date')
wti3=wti2.set_index('date')

wti3

In [ ]:
data = wti3.join(gpr1, how='inner')
data=data.dropna()
data

In [ ]:
pip install arch

In [ ]:
data['ret'] = np.log(data['Close'] / data['Close'].shift(1))
data=data.dropna()
data

In [ ]:
from arch import arch_model

model = arch_model(
    data['ret'] ,
    mean='Constant',
    vol='EGARCH',
    p=1, o=1, q=1,   # o = asymmetry term
    dist='normal'
)

res = model.fit()
print(res.summary())

# استخراج ولاتیلیتی شرطی
egarch_vol = res.conditional_volatility

# اضافه کردن به دیتا فریم
data = data.copy()
data["EGARCH_vol"] = egarch_vol

print(data[["ret", "EGARCH_vol"]].head())

In [ ]:
pip install pymc

In [ ]:
import pymc as pm
import numpy as np

returns = data['ret'].values

with pm.Model() as model:

    sigma = pm.Exponential("sigma", 2.0)
    phi = pm.Beta("phi", 20, 1)   # نزدیک به 1

    h = pm.AR("h", rho=phi, sigma=sigma, shape=len(returns))

    r = pm.Normal("r", mu=0, sigma=np.exp(h/2), observed=returns)

    trace = pm.sample(
        draws=200,
        tune=100,
        chains=4,
        target_accept=0.95)


In [ ]:
h_post = trace.posterior["h"]
volatility = np.exp(h_post / 2)
vol_mean = volatility.mean(dim=("chain","draw")).values
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(vol_mean)
plt.title("Stochastic Volatility (PyMC)")
plt.show()

In [ ]:
data["SV_vol"] = vol_mean
data

In [ ]:
import statsmodels.formula.api as smf

model = smf.quantreg('SV_vol ~ GPRD_THREAT + GPRD_ACT', data)
res = model.fit(q=0.9)
print(res.summary())

In [ ]:
data.to_csv("data.csv")

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations

In [ ]:
df = pd.read_csv("data.csv").dropna()
df = df.copy()
df=df.reset_index()
# 🔥 استفاده از ستون واقعی date (خیلی مهم)
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")

# حذف timezone اگر وجود دارد
df.index = df.index.tz_localize(None)

# مرتب‌سازی زمانی
df = df.sort_index()
y = df["YZ_vol"].values
X = df[["GPRD_THREAT", "GPRD_ACT"]].values

T, k = X.shape

In [ ]:
model_list = []

for i in range(1, k+1):
    for combo in combinations(range(k), i):
        model_list.append(list(combo))

M = len(model_list)


In [ ]:
alpha = 0.99      # model forgetting
lambda_ = 0.99    # parameter forgetting

weights = np.ones(M) / M

y_hat = np.zeros(T)
weights_time = np.zeros((T, M))

In [ ]:
theta = []
P = []

for m in range(M):
    p = len(model_list[m]) + 1
    theta.append(np.zeros((p,1)))
    P.append(np.eye(p) * 1000)

In [ ]:
for t in range(1, T):

    pred = np.zeros(M)
    loglik = np.zeros(M)

    for m in range(M):

        idx = model_list[m]

        x_tm1 = np.concatenate(([1], X[t-1, idx])).reshape(-1,1)

        th = theta[m]
        Pm = P[m]

        # prediction
        y_pred = float(x_tm1.T @ th)
        pred[m] = y_pred

        # error
        e = y[t] - y_pred

        # variance (adaptive)
        sigma2 = e**2 + 1e-6

        # Kalman gain (stable)
        S = lambda_ + (x_tm1.T @ Pm @ x_tm1)[0,0]
        K = (Pm @ x_tm1) / S

        # update theta
        th_new = th + K * e

        # Joseph form (numerical stability)
        I = np.eye(Pm.shape[0])
        P_new = (I - K @ x_tm1.T) @ Pm @ (I - K @ x_tm1.T).T + K * sigma2 * K.T
        P_new = P_new / lambda_

        theta[m] = th_new
        P[m] = P_new

        # log-likelihood (important!)
        loglik[m] = -0.5*(np.log(2*np.pi*sigma2) + (e**2)/sigma2)

    # 🔥 log-weight update (stable)
    logw = alpha * np.log(weights + 1e-12) + loglik
    logw = logw - np.max(logw)

    weights = np.exp(logw)
    weights = weights / np.sum(weights)

    # DMA prediction
    y_hat[t] = np.sum(weights * pred)
    weights_time[t,:] = weights

In [ ]:
importance = np.zeros((T, k))

for t in range(T):
    for j in range(k):
        importance[t,j] = np.sum(
            weights_time[t,:] *
            np.array([j in model for model in model_list])
        )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# =========================
# 1. تاریخ واقعی
# =========================

dates = pd.to_datetime(df.index)

# =========================
# 2. رسم نمودار
# =========================

plt.figure(figsize=(12,4))

var_names = ["GPRD_THREAT", "GPRD_ACT"]

for j in range(k):
    plt.plot(dates, importance[:, j], label=var_names[j])

# =========================
# 3. فرمت محور زمان
# =========================

plt.gcf().autofmt_xdate()

plt.title("Time-varying Importance (DMA)")
plt.xlabel("Time")
plt.ylabel("Importance")

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.tight_layout()
plt.savefig("figure.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# =========================
# 1. اطمینان از تاریخ درست
# =========================

dates = pd.to_datetime(df.index)

# =========================
# 2. رسم نمودار
# =========================

plt.figure(figsize=(12,5))

plt.plot(dates, y, label="Actual")
plt.plot(dates, y_hat, label="DMA", color="red")

# =========================
# 3. فرمت تاریخ محور X
# =========================

plt.gcf().autofmt_xdate()  # چرخش خودکار تاریخ‌ها

plt.title("DMA Forecast vs Actual")
plt.xlabel("Time")
plt.ylabel("Value")

plt.legend()
plt.tight_layout()
plt.savefig("figure1.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# =========================
# 1. ساخت صحیح DataFrame (اول این باید باشد)
# =========================

var_names = ["GPRD_THREAT", "GPRD_ACT"]

importance_df = pd.DataFrame(
    importance,
    columns=var_names,
    index=pd.to_datetime(df.index)
)

# حذف میلی‌ثانیه/نانوثانیه (تمیز کردن تاریخ)
importance_df.index = importance_df.index.floor('D')

# =========================
# 2. رسم heatmap
# =========================

plt.figure(figsize=(14,5))

ax = sns.heatmap(
    importance_df.T,
    cmap="viridis",
    cbar_kws={'label': 'Importance'},
    xticklabels=False   # جلوگیری از بهم‌ریختگی
)

# =========================
# 3. تنظیم محور X به تاریخ واقعی
# =========================

step = max(1, len(importance_df)//8)  # فقط 8 تا لیبل تمیز

plt.xticks(
    ticks=[i for i in range(0, len(importance_df), step)],
    labels=importance_df.index[::step].strftime('%Y-%m-%d'),
    rotation=45
)

# =========================
# 4. لیبل‌ها
# =========================

plt.title("Time-Varying Importance of Geopolitical Risk (DMA)")
plt.xlabel("Time")
plt.ylabel("Variables")

plt.tight_layout()
plt.savefig("figure2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 1. FIX تاریخ (درست)
# =========================================================

df = df.copy()
df = df.reset_index()

df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df.index = df.index.tz_localize(None)
df = df.sort_index()

# =========================================================
# 2. importance alignment
# =========================================================

importance = np.array(importance)

n = min(len(df), len(importance))

df = df.iloc[:n]
importance = importance[:n]

var_names = ["GPRD_THREAT", "GPRD_ACT"]

importance_df = pd.DataFrame(
    importance,
    columns=var_names,
    index=df.index
)

# =========================================================
# 3. crisis
# =========================================================

crises = {
    "GFC": ("2008-09-01", "2009-03-01"),
    "COVID": ("2020-02-01", "2020-06-01"),
    "Ukraine": ("2022-02-01", "2022-08-01"),
    "MidelEastWar": ("2026-02-28", "2026-04-27")
}

crises = {k: (pd.to_datetime(v[0]), pd.to_datetime(v[1])) for k,v in crises.items()}

# crisis mask
crisis_mask = np.zeros(len(importance_df), dtype=bool)

for k,(s,e) in crises.items():
    crisis_mask |= (importance_df.index >= s) & (importance_df.index <= e)

normal_df = importance_df[~crisis_mask]
crisis_df = importance_df[crisis_mask]

# =========================================================
# 4. PLOT (IMPORTANT FIX HERE)
# =========================================================

fig, ax = plt.subplots(2, 1, figsize=(14,6), sharey=True)

# ---------------- NORMAL ----------------
im1 = ax[0].imshow(
    normal_df.T.values,
    aspect='auto',
    cmap='viridis'
)

ax[0].set_title("Normal Period - DMA Importance")

ax[0].set_yticks(range(len(var_names)))
ax[0].set_yticklabels(var_names)

ax[0].set_xticks(np.linspace(0, len(normal_df)-1, 15))
ax[0].set_xticklabels(normal_df.index[np.linspace(0, len(normal_df)-1, 15, dtype=int)].strftime('%Y-%m'))

# ---------------- CRISIS ----------------
im2 = ax[1].imshow(
    crisis_df.T.values,
    aspect='auto',
    cmap='viridis'
)

ax[1].set_title("Crisis Period - DMA Importance")

ax[1].set_yticks(range(len(var_names)))
ax[1].set_yticklabels(var_names)

if len(crisis_df) > 0:
    ax[1].set_xticks(np.linspace(0, len(crisis_df)-1, 15))
    ax[1].set_xticklabels(crisis_df.index[np.linspace(0, len(crisis_df)-1, 15, dtype=int)].strftime('%Y-%m'))

# =========================================================
# 5. colorbar
# =========================================================

from mpl_toolkits.axes_grid1 import make_axes_locatable

divider = make_axes_locatable(ax[1])
divider1 = make_axes_locatable(ax[0])
cax = divider.append_axes("right", size="3%", pad=0.1)
cax1 = divider1.append_axes("right", size="3%", pad=0.1)
cbar = fig.colorbar(im1, cax=cax)
cbar1 = fig.colorbar(im2, cax=cax1)
cbar.set_label("Importance")
cbar1.set_label("Importance")
ax[1].set_xlabel("Time")

plt.tight_layout()
plt.savefig("figure3.png", dpi=300, bbox_inches='tight')
plt.show()